# Demo 13 — Cinque pattern da riconoscere

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
sr_pattern = 16000
t_pattern = np.arange(sr_pattern) / sr_pattern
rng_pattern = np.random.default_rng(21)
rumore_pattern = rng_pattern.standard_normal(len(t_pattern))
pattern = [
    ("Tono: una riga", np.sin(2*np.pi*600*t_pattern)),
    ("Nota: più armoniche", sum(np.sin(2*np.pi*250*k*t_pattern)/k for k in range(1, 7))),
    ("Chirp: traccia inclinata", signal.chirp(t_pattern, f0=300, f1=2600, t1=1)),
    ("Colpo: banda verticale", rumore_pattern*np.exp(-0.5*((t_pattern-0.5)/0.003)**2)),
    ("Rumore: energia diffusa", rumore_pattern),
]
pattern = [(nome, 0.4*x/np.max(abs(x))) for nome, x in pattern]
D_pattern = [al.analysis.stft(x, n_fft=512, hop_length=128) for _, x in pattern]
ref_pattern = max(np.max(abs(D)) for D in D_pattern)
fig = make_subplots(rows=2, cols=3, subplot_titles=[n for n, _ in pattern] + [""])
for k, ((nome, _), D) in enumerate(zip(pattern, D_pattern)):
    db = al.analysis.amplitude_to_db(abs(D), ref=ref_pattern, top_db=None)
    r, c = divmod(k, 3)
    fig.add_trace(go.Heatmap(x=np.arange(D.shape[1]) * 128 / sr_pattern,
                             y=np.arange(D.shape[0]) * sr_pattern / 512, z=db,
                             colorscale="Magma", zmin=-70, zmax=0,
                             showscale=(k == 4), colorbar=dict(title="dB")), row=r + 1, col=c + 1)
fig.update_yaxes(range=[0, 4000])
fig.update_layout(height=560, title="Cinque pattern da riconoscere")
fig.show()
for nome, x in pattern:
    print(nome); display(al.safe_audio(al.fade_edges(x, sr_pattern), sr_pattern, normalize=False))
